# Wan2.1 I2V 14B — API Server (GGUF Q4)
**Same engine as Isi-dev's notebook | Portrait 9:16 | ~26 min on T4**

Run all → wait for **API URL** → paste in Phase 6.

In [ ]:
# @title 📦 1. Install (same as Isi-dev)
!pip install torch==2.6.0 torchvision==0.21.0 -q
!pip install -q torchsde einops diffusers accelerate xformers==0.0.29.post2
!pip install -q av fastapi uvicorn nest-asyncio
!apt -y install -qq aria2 ffmpeg

import os, torch, gc, uuid, time, threading, asyncio, subprocess, random, sys, json
from pathlib import Path
from PIL import Image
import numpy as np
import imageio

In [ ]:
# @title 🤖 2. Setup ComfyUI + GGUF
%cd /content
!git clone -q https://github.com/Isi-dev/ComfyUI
%cd /content/ComfyUI/custom_nodes
!git clone -q https://github.com/Isi-dev/ComfyUI_GGUF.git
%cd /content/ComfyUI/custom_nodes/ComfyUI_GGUF
!pip install -q -r requirements.txt
%cd /content/ComfyUI

print("Downloading Q4 GGUF model...")
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/city96/Wan2.1-I2V-14B-480P-gguf/resolve/main/wan2.1-i2v-14b-480p-Q4_0.gguf -d /content/ComfyUI/models/unet -o wan2.1-i2v-14b-480p-Q4_0.gguf

print("Downloading support models...")
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors -d /content/ComfyUI/models/text_encoders -o umt5_xxl_fp8_e4m3fn_scaled.safetensors
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/vae/wan_2.1_vae.safetensors -d /content/ComfyUI/models/vae -o wan_2.1_vae.safetensors
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/clip_vision/clip_vision_h.safetensors -d /content/ComfyUI/models/clip_vision -o clip_vision_h.safetensors
print("✅ All models ready!")

In [ ]:
# @title 🚀 3. Start API Server
sys.path.insert(0, '/content/ComfyUI')

from comfy import model_management
from nodes import (
    CLIPLoader, CLIPTextEncode, VAEDecode, VAELoader,
    KSampler, LoadImage, CLIPVisionLoader, CLIPVisionEncode
)
from custom_nodes.ComfyUI_GGUF.nodes import UnetLoaderGGUF
from comfy_extras.nodes_model_advanced import ModelSamplingSD3
from comfy_extras.nodes_wan import WanImageToVideo

unet_loader = UnetLoaderGGUF()
model_sampling = ModelSamplingSD3()
clip_loader = CLIPLoader()
clip_encode_pos = CLIPTextEncode()
clip_encode_neg = CLIPTextEncode()
vae_loader = VAELoader()
clip_vision_loader = CLIPVisionLoader()
clip_vision_encode = CLIPVisionEncode()
load_image = LoadImage()
wan_image_to_video = WanImageToVideo()
ksampler = KSampler()
vae_decode = VAEDecode()

def clear_memory():
    gc.collect(); torch.cuda.empty_cache()

def gen_video(image_path, prompt, seed, width, height, frames=49, steps=20):
    with torch.inference_mode():
        clip = clip_loader.load_clip("umt5_xxl_fp8_e4m3fn_scaled.safetensors", "wan", "default")[0]
        pos = clip_encode_pos.encode(clip, prompt)[0]
        neg = clip_encode_neg.encode(clip, "色调艳丽，过曝，静态，模糊，最差质量，低质量，丑陋")[0]
        del clip; torch.cuda.empty_cache()
        img = load_image.load_image(image_path)[0]
        cv = clip_vision_loader.load_clip("clip_vision_h.safetensors")[0]
        cv_out = clip_vision_encode.encode(cv, img, "none")[0]
        del cv; torch.cuda.empty_cache()
        vae = vae_loader.load_vae("wan_2.1_vae.safetensors")[0]
        p_out, n_out, lat = wan_image_to_video.encode(pos, neg, vae, width, height, frames, 1, img, cv_out)
        model = unet_loader.load_unet("wan2.1-i2v-14b-480p-Q4_0.gguf")[0]
        model = model_sampling.patch(model, 8)[0]
        samp = ksampler.sample(model=model, seed=seed, steps=steps, cfg=1.0,
            sampler_name="uni_pc", scheduler="simple",
            positive=p_out, negative=n_out, latent_image=lat)[0]
        del model; torch.cuda.empty_cache()
        dec = vae_decode.decode(vae, samp)[0]
        del vae; torch.cuda.empty_cache()
        return [(f.cpu().numpy() * 255).astype(np.uint8) for f in dec]

import nest_asyncio; nest_asyncio.apply()
!pkill -f cloudflared 2>/dev/null; sleep 1
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /tmp/cloudflared
!chmod +x /tmp/cloudflared

from fastapi import FastAPI, File, UploadFile, Form
from fastapi.responses import FileResponse
import uvicorn

app = FastAPI(); tasks = {}; API_URL = None

def start_tunnel():
    global API_URL
    proc = subprocess.Popen(["/tmp/cloudflared", "tunnel", "--url", "http://localhost:8080"],
        stdout=subprocess.DEVNULL, stderr=subprocess.PIPE)
    for line in iter(proc.stderr.readline, b''):
        t = line.decode().strip()
        if "trycloudflare.com" in t:
            API_URL = "https://" + t.split("https://")[-1].strip()
            print(f"\n✅  API URL: {API_URL}\n")
            break

@app.get("/health")
def health():
    return {"status": "ok", "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}

@app.post("/generate")
async def generate(image: UploadFile = File(...), prompt: str = Form(...)):
    task_id = uuid.uuid4().hex[:8]
    img = Image.open(image.file).convert("RGB")
    img = img.resize((480, 848))
    p = f"/content/in_{task_id}.png"
    img.save(p)
    tasks[task_id] = {"state": "queued"}
    threading.Thread(target=_run, args=(task_id, prompt, p), daemon=True).start()
    return {"task_id": task_id}

def _run(task_id, prompt, img_path):
    tasks[task_id]["state"] = "running"
    try:
        f = gen_video(img_path, prompt, random.randint(0,2**32), 480, 848, 49, 20)
        out = f"/content/out_{task_id}.mp4"
        with imageio.get_writer(out, fps=16) as w:
            for fr in f: w.append_data(fr)
        tasks[task_id]["state"] = "completed"
        tasks[task_id]["video_url"] = f"{API_URL}/video/{task_id}"
    except Exception as e:
        tasks[task_id]["state"] = "failed"
        tasks[task_id]["error"] = str(e)

@app.get("/status/{task_id}")
def status(task_id: str):
    return tasks.get(task_id, {"state": "unknown"})

@app.get("/video/{task_id}")
def video(task_id: str):
    p = f"/content/out_{task_id}.mp4"
    if os.path.exists(p): return FileResponse(p, media_type="video/mp4")
    return {"error": "not found"}

threading.Thread(target=start_tunnel, daemon=True).start()
time.sleep(3)
config = uvicorn.Config(app=app, host="0.0.0.0", port=8080, log_level="error")
loop = asyncio.get_event_loop()
loop.create_task(uvicorn.Server(config).serve())
print("✅ Server running!")
while API_URL is None: time.sleep(1)

Copy **API URL** → paste in Phase 6. Keep this tab open.